# Feature-based TabPFN classification

Fold-based cross-validation of TabPFN on the 60 summary features, for every burst length (B1–B4) and correction type (uncorrected, rotdaily, rotbasal). Folds come from `indices_cv_fold_lookup.csv` (`04_build_cv_fold_lookup`): train = `fold != f`, test = `fold == f`, and all sub-windows of a parent burst share a fold.

Each run writes three outputs for the behaviour target (the activity target is also fitted, but its outputs are not saved; see the save cell):
- per-fold metrics (`*_features_dl_tabpfn_metrics.csv`)
- per-fold confusion matrices (`*_features_dl_tabpfn_confusion.csv`)
- per-window out-of-fold predictions with class probabilities (`*_features_dl_tabpfn_oof_predictions.csv`), carrying `Ind_ID`, `burst_id` and `fold`

Metrics are computed with the same `compute_metrics` helper as `05a`, `05b` and `05d`.

In [ ]:
import glob
import os
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import LabelEncoder
from tabpfn import TabPFNClassifier

## Shared helpers
Identical in `05a`, `05b`, `05c` and `05d`, so every model family uses the same metric definitions and output format.

In [ ]:
N_FEATURES = 60  # guards against ID columns leaking into X
TEST_ID_COLS = ["Ind_ID", "new_burst", "burst_id", "fold"]
LEAD_COLS = ["burst", "correction_type", "target", "fold", "model"]


def compute_metrics(y_true, y_pred, proba, n_classes, is_binary):
    """Identical metric definitions for every model family.

    Args:
        y_true (np.ndarray): Encoded true labels
        y_pred (np.ndarray): Encoded predicted labels
        proba (np.ndarray): Predicted class probabilities, one column per class
        n_classes (int): Number of classes in the target
        is_binary (bool): Whether this is binary classification

    Returns:
        dict: Global metrics (weighted and macro averages)
    """
    labels = np.arange(n_classes)
    try:
        roc = (
            roc_auc_score(y_true, proba[:, 1])
            if is_binary
            else roc_auc_score(
                y_true, proba, multi_class="ovr", average="weighted", labels=labels
            )
        )
    except ValueError:
        roc = np.nan
    return {
        "roc_auc": roc,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "f1_score": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "precision": precision_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_macro": f1_score(
            y_true, y_pred, average="macro", labels=labels, zero_division=0
        ),
        "precision_macro": precision_score(
            y_true, y_pred, average="macro", labels=labels, zero_division=0
        ),
        "recall_macro": recall_score(
            y_true, y_pred, average="macro", labels=labels, zero_division=0
        ),
    }


def confusion_rows(y_true, y_pred, class_names, model_name):
    """Fixed CxC confusion matrix in long format, so folds can be pooled."""
    n = len(class_names)
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(n))
    return [
        {
            "model": model_name,
            "actual_label": class_names[i],
            "predicted_label": class_names[j],
            "count": int(cm[i, j]),
        }
        for i in range(n)
        for j in range(n)
    ]


def prediction_rows(test_ids, y_true, y_pred, proba, class_names, model_name):
    """One row per test window: IDs, labels and class probabilities."""
    out = test_ids.reset_index(drop=True).copy()
    out.insert(0, "model", model_name)
    names = np.asarray(class_names, dtype=object)
    out["actual_label"] = names[np.asarray(y_true, dtype=int)]
    out["predicted_label"] = names[np.asarray(y_pred, dtype=int)]
    for i, cls in enumerate(class_names):
        out[f"prob_{cls}"] = proba[:, i]
    return out


def finalise(frames, lead_cols):
    """Concatenate per-fold frames and move the identifying columns to the front."""
    frames = [d for d in frames if d is not None and not d.empty]
    if not frames:
        return pd.DataFrame(columns=lead_cols)
    df = pd.concat(frames, ignore_index=True)
    return df[lead_cols + [c for c in df.columns if c not in lead_cols]]


## Model functions

In [ ]:
def features_tabpfn_model(
    X_train,
    X_test,
    y_train,
    y_test,
    class_names,
    test_ids,
    use_class_weight=True,
    is_binary=False,
):
    """Train and evaluate TabPFN on one CV fold.

    Args:
        X_train (pd.DataFrame): Training features
        X_test (pd.DataFrame): Test features
        y_train (np.ndarray): Encoded training labels
        y_test (np.ndarray): Encoded test labels
        class_names (list): Original class names, ordered by encoded label
        test_ids (pd.DataFrame): TEST_ID_COLS for the test windows, in X_test order
        use_class_weight (bool): Whether to correct for class imbalance via
            TabPFN's balance_probabilities option. TabPFN is a frozen,
            pretrained in-context-learning model: .fit() only caches the
            training rows as context and runs no gradient descent, so there is
            no class_weight/sample_weight parameter like the other pipelines.
            balance_probabilities instead rescales predict_proba's output by
            the empirical training class frequencies at inference time.
        is_binary (bool): Whether this is binary classification

    Returns:
        tuple: (metrics_df, confusion_df, predictions_df)
    """
    n_classes = len(class_names)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        t0 = time.time()
        model = TabPFNClassifier(
            random_state=42, balance_probabilities=use_class_weight
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        proba = model.predict_proba(X_test)
        metrics = {
            "model": "TabPFN",
            **compute_metrics(y_test, y_pred, proba, n_classes, is_binary),
            "time": time.time() - t0,
        }
    print("TabPFN Done")
    return (
        pd.DataFrame([metrics]),
        pd.DataFrame(confusion_rows(y_test, y_pred, class_names, "TabPFN")),
        prediction_rows(test_ids, y_test, y_pred, proba, class_names, "TabPFN"),
    )

In [ ]:
ID_COLS = ["Ind_ID", "new_burst", "burst_id", "behaviour", "activity", "fold"]


def process_parquet_file(file_path, fold_lookup, use_class_weight=True):
    """Run fold-based CV on one parquet file for both the binary (activity)
    and multiclass (behaviour) targets.

    Args:
        file_path (str): Path to the parquet file
        fold_lookup (pd.DataFrame): Fold lookup table with columns
            burst, Ind_ID, new_burst, burst_id, behaviour, activity, fold
        use_class_weight (bool): Whether to balance TabPFN's output
            probabilities via balance_probabilities (see features_tabpfn_model)

    Returns:
        dict: binary_/multiclass_ metrics, conf and pred DataFrames
    """
    burst = os.path.basename(os.path.dirname(file_path))
    correction_type = os.path.basename(file_path).split("_")[-1].split(".")[0]
    print(f"Processing {correction_type} for {burst}")

    features = pd.read_parquet(file_path)  # features + Ind_ID + new_burst
    m = fold_lookup.loc[fold_lookup["burst"] == burst.lower(), ID_COLS]
    df = features.merge(m, on=["Ind_ID", "new_burst"], how="inner").reset_index(
        drop=True
    )  # inner drops Remove/unlabelled
    X = df.drop(columns=ID_COLS)
    assert X.shape[1] == N_FEATURES, f"Unexpected feature columns: {list(X.columns)}"
    folds = sorted(df["fold"].unique())

    out = {t: {"metrics": [], "conf": [], "pred": []} for t in ("binary", "multiclass")}
    for is_binary, col, tgt, key in [
        (True, "activity", "Activity", "binary"),
        (False, "behaviour", "Behaviour", "multiclass"),
    ]:
        enc = LabelEncoder()
        y_enc = enc.fit_transform(df[col])
        # Encode "Active" as 1 so binary ROC AUC uses the active class
        if (
            is_binary
            and set(enc.classes_) == {"Active", "Inactive"}
            and enc.transform(["Active"])[0] != 1
        ):
            enc.classes_ = enc.classes_[::-1]
            y_enc = enc.transform(df[col])
        class_names = list(enc.classes_)

        for f in folds:
            te = (df["fold"] == f).to_numpy()
            met, conf, pred = features_tabpfn_model(
                X[~te],
                X[te],
                y_enc[~te],
                y_enc[te],
                class_names,
                test_ids=df.loc[te, TEST_ID_COLS],
                use_class_weight=use_class_weight,
                is_binary=is_binary,
            )
            for d in (met, conf, pred):
                d["correction_type"] = correction_type
                d["burst"] = burst
                d["target"] = tgt
                d["fold"] = f
            out[key]["metrics"].append(met)
            out[key]["conf"].append(conf)
            out[key]["pred"].append(pred)

    return {
        f"{key}_{kind}": finalise(out[key][kind], LEAD_COLS)
        for key in ("binary", "multiclass")
        for kind in ("metrics", "conf", "pred")
    }

In [ ]:
def wrap_run_feat_tabpfn(
    folder_location,
    fold_id_path="../data/temp/indices_cv_fold_lookup.csv",
    correction_filters=None,
    use_class_weight=True,
):
    """Process all parquet files in a folder.

    Args:
        folder_location (str): Path to folder containing parquet files
        fold_id_path (str): Path to CSV file containing fold IDs
        correction_filters (list, optional): List of correction types to include
        use_class_weight (bool): Whether to balance TabPFN's output
            probabilities via balance_probabilities

    Returns:
        dict: Dictionary with combined results DataFrames
    """
    filetype = folder_location.split("/")[-2]
    print(f"TabPFN training started for {filetype}")

    # Find all parquet files
    all_files = [
        os.path.normpath(f).replace("\\", "/")
        for f in glob.glob(folder_location + "**/*.parquet", recursive=True)
    ]

    # Filter by correction type
    if correction_filters is not None:
        all_files = [
            f
            for f in all_files
            if any(corr.lower() in f.lower() for corr in correction_filters)
        ]

    print(f"Found {len(all_files)} parquet files")

    fold_lookup = pd.read_csv(fold_id_path)

    # Process all files and combine results
    all_results = [
        process_parquet_file(f, fold_lookup, use_class_weight=use_class_weight)
        for f in all_files
    ]
    keys = [
        f"{t}_{k}"
        for t in ("binary", "multiclass")
        for k in ("metrics", "conf", "pred")
    ]
    return {k: pd.concat([r[k] for r in all_results], ignore_index=True) for k in keys}


## Run over all bursts and correction types

In [ ]:
fold_id_path = "../data/temp/indices_cv_fold_lookup.csv"
correction_types_to_process = ["uncorrected", "rotdaily", "rotbasal"]
folder_locations = [
    "../data/raw/features/Burst_1/",
    "../data/raw/features/Burst_2/",
    "../data/raw/features/Burst_3/",
    "../data/raw/features/Burst_4/",
]

In [ ]:
# Run the TabPFN training for all files in the specified folders
all_burst_results = [
    wrap_run_feat_tabpfn(
        folder, fold_id_path, correction_filters=correction_types_to_process
    )
    for folder in folder_locations
]

# Combine bursts into single dataframes
res = {
    k: pd.concat([r[k] for r in all_burst_results], ignore_index=True)
    for k in all_burst_results[0]
}


## Save

In [ ]:
# Output to CSV files
out_files = {
    # Activity (active/inactive) outputs are not used in the paper and are not
    # shipped; uncomment to save them (create data/output/comparison_activity/ first)
    # "binary_metrics": "comparison_activity/activity_features_dl_tabpfn_metrics.csv",
    # "binary_conf": "comparison_activity/activity_features_dl_tabpfn_confusion.csv",
    # "binary_pred": "comparison_activity/activity_features_dl_tabpfn_oof_predictions.csv",
    "multiclass_metrics": "comparison_behaviour/behaviour_features_dl_tabpfn_metrics.csv",
    "multiclass_conf": "comparison_behaviour/behaviour_features_dl_tabpfn_confusion.csv",
    "multiclass_pred": "comparison_behaviour/behaviour_features_dl_tabpfn_oof_predictions.csv",
}
for k, path in out_files.items():
    res[k].to_csv(f"../data/output/{path}", index=False)
    print(f"Saved {len(res[k])} rows -> {path}")

## Class-weight ablation (B1 and B2 uncorrected)
Re-runs TabPFN on B1 uncorrected (the raw-data baseline) and B2 uncorrected (the final classifier's burst length) with `use_class_weight=False`, i.e. without `balance_probabilities`. Same seed, folds and data as the main run, so the only difference is the class balancing. These are the same two datasets as the LOIO validation (`08a`–`08c`); see `05a` for RF/XGB and `05d` for HR. Run the imports, helper and function cells first; these cells do not need the main run in memory. Only the behaviour target is kept (the activity target is also fitted inside `process_parquet_file`), and the outputs go to separate `*_nocw_*` files, so the main-run files are untouched.

In [ ]:
# Run TabPFN without balance_probabilities on B1 and B2 uncorrected
ABLATION_BURSTS = ["Burst_1", "Burst_2"]

fold_lookup = pd.read_csv("../data/temp/indices_cv_fold_lookup.csv")
nocw = [
    process_parquet_file(
        f"../data/raw/features/{b}/annotated_features_{b.lower()}_uncorrected.parquet",
        fold_lookup,
        use_class_weight=False,
    )
    for b in ABLATION_BURSTS
]

# Behaviour target only
nocw_res = {
    k: pd.concat([r[f"multiclass_{k}"] for r in nocw], ignore_index=True)
    for k in ("metrics", "conf", "pred")
}
print(nocw_res["pred"].groupby("burst").size())

In [ ]:
# Compare pooled per-behaviour precision, recall and predicted/true ratio
# with the balanced main run (read from the saved OOF file), per burst
cw_pred = pd.read_csv(
    "../data/output/comparison_behaviour/behaviour_features_dl_tabpfn_oof_predictions.csv"
)
cw_pred = cw_pred[
    cw_pred["burst"].isin(ABLATION_BURSTS)
    & (cw_pred["correction_type"] == "uncorrected")
]


def pooled_pr(pred):
    """Pooled precision, recall and predicted/true ratio per behaviour."""
    rows = []
    for cls in sorted(pred["actual_label"].unique()):
        tp = ((pred["actual_label"] == cls) & (pred["predicted_label"] == cls)).sum()
        n_true = (pred["actual_label"] == cls).sum()
        n_pred = (pred["predicted_label"] == cls).sum()
        rows.append(
            {
                "behaviour": cls,
                "n_true": n_true,
                "precision": tp / n_pred if n_pred else np.nan,
                "recall": tp / n_true,
                "pred_true": n_pred / n_true,
            }
        )
    return pd.DataFrame(rows).set_index("behaviour")


pr_comp = pd.concat(
    {
        b: pooled_pr(cw_pred[cw_pred["burst"] == b]).join(
            pooled_pr(nocw_res["pred"][nocw_res["pred"]["burst"] == b]).drop(
                columns="n_true"
            ),
            lsuffix="_cw",
            rsuffix="_nocw",
        )
        for b in ABLATION_BURSTS
    },
    names=["burst"],
)
pr_comp[
    [
        "n_true",
        "precision_cw",
        "precision_nocw",
        "recall_cw",
        "recall_nocw",
        "pred_true_cw",
        "pred_true_nocw",
    ]
].round(3)

In [ ]:
# Output to CSV files (separate from the main-run files)
for k in ("metrics", "conf", "pred"):
    name = {"metrics": "metrics", "conf": "confusion", "pred": "oof_predictions"}[k]
    path = f"comparison_behaviour/behaviour_features_dl_tabpfn_nocw_{name}.csv"
    nocw_res[k].to_csv(f"../data/output/{path}", index=False)
    print(f"Saved {len(nocw_res[k])} rows -> {path}")